In [ ]:
###########################################################################
## Basic
###########################################################################
%load_ext autoreload
%autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("""<style>div.output_area{max-height:10000px;overflow:scroll;}</style>"""))


###########################################################################
## Warnings
###########################################################################
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning) 
warnings.filterwarnings("ignore", category=FutureWarning) 


###########################################################################
## Utils
###########################################################################
from timeUtils import timestat
from listUtils import getFlatList
from fileIO import fileIO
from matchUtils import getUnmatchedArtistsFromDB, getArtistsToMatch, serialRun, multiProcRun


###########################################################################
## DB
###########################################################################
from masterManualEntries import masterManualEntries
from masterArtistNameDB import masterArtistNameDB
from masterArtistMerger import masterArtistMerger
from masterMultiArtistDB import masterMultiArtistDB
from masterArtistNameCreator import masterArtistNameCreator
from convertByteString import convertByteString
from musicDBMap import musicDBMap
from matchRunParams import matchRunParams
from masterDBGate import masterDBGate
from masterDBData import masterDBData
from matchDBData import matchDBData
from masterMatchGate import masterMatchGate
from matchArtistToDB import matchArtistToDB


###########################################################################
## General
###########################################################################
from pandas import isna, notna, Series, DataFrame, concat
from copy import deepcopy
from uuid import uuid4


###########################################################################
## Sys
###########################################################################
import sys
print("Python: {0}".format(sys.version))

# Initialize Data Access

In [ ]:
mme        = masterManualEntries()
mmeDF      = mme.getDataFrame()
mdbGate    = masterDBGate()
cbs        = convertByteString()
mam        = masterArtistMerger()
mma        = masterMultiArtistDB()
manDB      = masterArtistNameDB("main")
multimanDB = masterArtistNameDB("multi")
transDB    = masterArtistNameDB("translation")
mdbGate    = masterDBGate()
io         = fileIO()

# Global Variables For Matching

In [ ]:
dbToUse         = "AllMusic" # 8
dbToUse         = "RateYourMusic"
#dbToUse         = "Genius"
#dbToUse         = "MusicBrainz" # 11
#dbToUse         = "DeezerAPI"

artistRunVars   = {}
artistRunVars["minArtistAlbums"] = 6
artistRunVars["maxArtistAlbums"] = None
artistRunVars["shuffleArtists"]  = False
artistRunVars["maxNumArtists"]   = 15000

numAlbumsReq={dbToUse: max([int(artistRunVars["minArtistAlbums"]/2), 3])}

usePool = True
dtype   = "Search"

numProcs = 3

In [ ]:
##################################################################################################################
# Load Artists/Albums Data For 'dbToUse'
##################################################################################################################
ts = timestat("Getting ArtistName/NumAlbums Lookup Data For [{0}] DB".format(dbToUse))
mdbDataDBToUse    = masterDBData(dtype=dtype, dbs=dbToUse)
mdbDataDBToUse.loadArtists(numAlbumsReq=numAlbumsReq)
mdbDataDBToUse.loadAlbums()
artistDBToUseData = mdbDataDBToUse.getDBBasicInfo(dbToUse)
ts.stop()


##################################################################################################################
# Find Artists To Match Based On Global Variables
##################################################################################################################
ts = timestat("Collecting Unmatched DB Artists For [{0}]".format(dbToUse))
unmatchedArtists   = getUnmatchedArtistsFromDB(dbToUse, mdbDataDBToUse, mam, mmeDF)["dbArtistsToFind"]
artistsToMatch     = getArtistsToMatch(unmatchedArtists, artistRunVars)
artistsToMatchData = artistDBToUseData.loc[artistsToMatch.index].sort_values(by="NumAlbums", ascending=False)
ts.stop()
print("Found {0} Artists To Match For DB [{1}]".format(len(artistsToMatch), dbToUse))


##################################################################################################################
# Cross Check Albums Data
##################################################################################################################
mismatch = artistsToMatchData[~(artistsToMatchData["NumAlbums"] == mdbDataDBToUse.mdbData[dbToUse]["IDToAlbums"].loc[artistsToMatchData.index].apply(len))]
if mismatch.shape[0] > 0:
    raise ValueError("There is a mismatch between numAlbums and the length of the albums list")

print("\t{0: <20}: {1}".format("Artists",artistsToMatchData.shape[0]))
print("\t{0: <20}: {1}".format("Max Albums",artistsToMatchData["NumAlbums"].max()))
print("\t{0: <20}: {1}".format("Min Albums",artistsToMatchData["NumAlbums"].min()))
artistsToMatchData.head(3)

# Define Run

In [ ]:
%autoreload
chartTypes      = ['Discogs', 'MusicBrainz', 'AllMusic', 'Genius', 'DeezerAPI', 'RateYourMusic', 'KWorbiTunes', 'KWorbSpotify', 'SpotifyCharts']
dbsToMatch      = chartTypes

mrp = matchRunParams()
mmg = masterMatchGate(mdbDataDBToUse, dbToUse)
mmg.setArtistToMatchData(artistsToMatchData)
matchData = matchDBData(mmeDF=None, dtype=dtype, dbs=dbsToMatch)
matchData.loadArtists(Discogs=numAlbumsReq.get("Discogs",4),         MusicBrainz=numAlbumsReq.get("MusicBrainz",2), 
                      AllMusic=numAlbumsReq.get("AllMusic",3),       RateYourMusic=numAlbumsReq.get("RateYourMusic",2), 
                      Genius=numAlbumsReq.get("Genius",2),           AlbumOfTheYear=numAlbumsReq.get("AlbumOfTheYear",2), 
                      KWorbiTunes=numAlbumsReq.get("KWorbiTunes",2), KWorbSpotify=numAlbumsReq.get("KWorbiTunes",2),
                      DeezerAPI=numAlbumsReq.get("DeezerAPI",2),     LastFMAPI=numAlbumsReq.get("LastFMAPI",2),
                      SpotifyCharts=numAlbumsReq.get("SpotifyCharts",2))

# Match Artist Data

## Match Artist Names

In [ ]:
tsMatch = timestat("Matching [{0}] Artists From [{1}]".format(artistsToMatchData.shape[0], dbToUse))

In [ ]:
numRunData = mmg.setArtistRunData()
tsName = timestat("Matching Artist Names")
print("  Matching {0} Artists".format(numRunData))
params={"artistCutoff": 0.85, "by": "Artist", 'matchData': matchData}
runResults = multiProcRun(mmg, numProcs=numProcs, params=params) if usePool else serialRun(mmg=mmg, params=params)
mmg.setMatchedArtistData(runResults)
mmg.updateArtistDataArtistRunStatus()
tsName.stop()
 
tsMatch.update()

## Load Albums For Matched Names

In [ ]:
tsIDX = timestat("Loading Album Data For Matched Index")
matchData.loadAlbums(idxReq=mmg.matchedArtistIDXReq)
tsIDX.stop()
tsMatch.update()

## Match Artist Albums

In [ ]:
%autoreload
from matchArtistToDB import matchArtistToDB
tsAlbum = timestat("Matching Artist Albums")
for runNum,(runDefs, runThresholds) in enumerate(mrp.getRunParams()):
    #print("="*125)
    #print("Run #: {0}".format(runNum))
    runThresholds['albumCutoff'] = 0.7
    runThresholds['numAlbums']   = 1
    runThresholds['score']       = 0.0
    runThresholds["by"]          = "Album"
    runThresholds['detail']      = 0
    runThresholds['matchData']   = matchData
    runStatus = True
    while runStatus is True:
        numArtistsToMatchForRun = mmg.setAlbumRunData(**runDefs)
        if numArtistsToMatchForRun["ForRun"] <= 0:
            runStatus = False
            continue
            
        #print("  Matching {0} Artists For Run Number [{1}]".format(numArtistsToMatchForRun, runNum))        
        runResults = multiProcRun(mmg=mmg, numProcs=numProcs, params=runThresholds) if usePool else serialRun(mmg=mmg, params=runThresholds)
        mmg.setMatchedArtistAlbumData(runResults)
        mmg.updateArtistDataAlbumRunStatus()
tsAlbum.stop()
print("")

# Cross Match Results

## Create Cross Match Data

In [ ]:
matchedArtistAlbumData  = {item["ArtistID"]: item["Results"] for item in mmg.matchedArtistAlbumData}

dbCrossMatchIDs         = {}
artistsToCrossMatchData = {}
artistsToSelfMatchData  = {}

for artistID,artistIDData in matchedArtistAlbumData.items():
    for db,dbMatchData in artistIDData.items():
        if dbMatchData is None:
            continue
        if artistsToCrossMatchData.get(db) is None:
            artistsToCrossMatchData[db] = {}
        for dbID,dbIDMatchData in dbMatchData.items():
            if dbIDMatchData is not None:
                artistsToCrossMatchData[db][dbID] = {k: v for k,v in dbIDMatchData.items() if k in ["ArtistName", "NumAlbums"]}
            
#artistsToSelfMatchData  = {db: DataFrame(crossMatchData).T for db,crossMatchData in artistsToSelfMatchData.items()}
artistsToCrossMatchData = {db: DataFrame(crossMatchData).T for db,crossMatchData in artistsToCrossMatchData.items()}

In [ ]:
print("{0: <18}{1: <8}{2: <8}".format("DB","Match","Initial"))
for db,crossMatchData in artistsToCrossMatchData.items():
    print("{0: <18}{1: <8}{2: <8}".format(db,crossMatchData.shape[0],artistsToMatchData.shape[0]))

## Measure Number of Missing Self-matches

In [ ]:
############################################################################################################################################################
# Artist Name Matches
############################################################################################################################################################
print('='*50,"Missing Name Self-matches",'='*50)
madDF = mmg.matchedArtistData.apply(Series)
madDF = madDF.join(madDF["Results"].apply(Series))
madDF = madDF.drop(["Results"], axis=1)
for idx,row in madDF[madDF.apply(lambda x: x[dbToUse].get(x["ArtistID"]) is None, axis=1)][["ArtistID", dbToUse]].iterrows():
    print(idx,row["ArtistID"])
    for matchID,matchIDData in row[dbToUse].iteritems():
        print("\t",matchID,matchIDData)

## Copy Previous DBs

In [ ]:
matchDataPrimary      = deepcopy(matchData)
mdbDataDBToUsePrimary = deepcopy(mdbDataDBToUse)

## Cross Match

In [ ]:
tsCrossMatch = timestat("Cross Matching Artists") # [{0}] Artists From [{1}]".format(artistsToMatchData.shape[0], dbToUse))

matchData = matchDBData(mmeDF=None, dtype=dtype, dbs=dbToUse)
matchData.mdbData.setMDBData(mdbDataDBToUsePrimary.mdbData)
matchData.mdbData.prepareSearchData(None)

mrp = matchRunParams()

crossMatchResults = {}

################################################################################################################
# Loop Over DBs Matched In Previous Run
################################################################################################################
for dbToUseCrossMatch,dbArtistsToMatchData in artistsToCrossMatchData.items():
    
    ############################################################################################################
    # DB Setup
    ############################################################################################################
    mdbDataDBToUse = masterDBData(dtype=dtype, dbs=dbToUseCrossMatch)
    mdbDataDBToUse.setMDBData(matchDataPrimary.mdbData.mdbData)
    mmgCross = masterMatchGate(mdbDataDBToUse, dbToUseCrossMatch)
    mmgCross.setArtistToMatchData(dbArtistsToMatchData)
    
    
    ############################################################################################################
    # Match Artist Names
    ############################################################################################################
    numRunData = mmgCross.setArtistRunData()
    tsName = timestat("Matching {0} Artist Names For [{1}]".format(numRunData, dbToUseCrossMatch))
    params={"artistCutoff": 0.85, "by": "Artist", "matchData": matchData}
    runResults = multiProcRun(mmg=mmgCross, numProcs=numProcs, params=params) if usePool else serialRun(mmg=mmgCross, params=params)
    mmgCross.setMatchedArtistData(runResults)
    mmgCross.updateArtistDataArtistRunStatus()
    tsName.stop()
    tsCrossMatch.update()
    
    
    ############################################################################################################
    # Set Indices For Album Lookup
    ############################################################################################################
    #tsIDX = timestat("Loading Album Data For Matched Index")
    #matchData.loadAlbums(idxReq=mmgCross.matchedArtistIDXReq)
    #tsIDX.stop()
    tsCrossMatch.update()
    
        
    ############################################################################################################
    # Match Artist Albums
    ############################################################################################################
    tsAlbum = timestat("Matching Artist Albums For [{0}]".format(dbToUseCrossMatch))
    for runNum,(runDefs, runThresholds) in enumerate(mrp.getRunParams()):
        #runThresholds['albumCutoff'] = 0.7
        #runThresholds['numAlbums'] = 1
        #runThresholds['score'] = 0.0
        runThresholds["by"]        = "Album"
        runThresholds['detail']    = 0
        runThresholds['matchData'] = matchData

        runStatus = True
        while runStatus is True:
            numArtistsToMatchForRun = mmgCross.setAlbumRunData(**runDefs)
            if numArtistsToMatchForRun["ForRun"] <= 0:
                runStatus = False
                continue
                
            ##########################
            # Run Data For Diagnostics
            ##########################
            runData = mmgCross.getRunData()

            #print(numArtistsToMatchForRun)
            #print(mmgCross.getRunData().shape)
            #print(mmgCross.getRunData())
            runResults = multiProcRun(mmg=mmgCross, numProcs=numProcs, params=runThresholds) if usePool else serialRun(mmg=mmgCross, params=runThresholds)
            mmgCross.setMatchedArtistAlbumData(runResults)
            mmgCross.updateArtistDataAlbumRunStatus()
    tsAlbum.stop()
    crossMatchResults[dbToUseCrossMatch] = mmgCross
    print("")
tsCrossMatch.stop()

# Cross Match Results

## Cross Match Data

In [ ]:
def getAlbumMatchResults(matchedArtistAlbumData):
    primaryResults   = {}
    secondaryResults = {}
    tertiatyResults  = {}
    for dbToUseID,dbToUseAlbumMatchData in matchedArtistAlbumData.items():
        primaryResults[dbToUseID]   = {}
        secondaryResults[dbToUseID] = {}
        tertiatyResults[dbToUseID]  = {}
        for db,dbAlbumMatchData in dbToUseAlbumMatchData.items():
            if dbAlbumMatchData is None:
                continue
            albumResult = DataFrame(dbAlbumMatchData).T
            dbIDMatch   = albumResult[(albumResult["Match"] >= 2.0) & (albumResult["Tight"] >= 1.0)].sort_values(by=["Match","Tight"], ascending=False).head(1)
            if dbIDMatch.shape[0] > 0:
                primaryResults[dbToUseID][db]   = dbIDMatch.index[0]
                secondaryResults[dbToUseID][db] = dbIDMatch.index[0]
                tertiatyResults[dbToUseID][db]  = dbIDMatch.index[0]
            elif dbIDMatch.shape[0] == 0:
                dbIDMatch   = albumResult[(albumResult["Match"] >= 2.0)].sort_values(by=["Match"], ascending=False).head(1)
                if dbIDMatch.shape[0] > 0:
                    secondaryResults[dbToUseID][db] = dbIDMatch.index[0]
                    tertiatyResults[dbToUseID][db]  = dbIDMatch.index[0]
                elif dbIDMatch.shape[0] == 0:
                    dbIDMatch   = albumResult[(albumResult["Loose"] >= 2.0)].sort_values(by=["Match"], ascending=False).head(1)
                    if dbIDMatch.shape[0] > 0:
                        tertiatyResults[dbToUseID][db]  = dbIDMatch.index[0]
                
    primaryResults   = Series(primaryResults).apply(Series)
    secondaryResults = Series(secondaryResults).apply(Series)
    tertiatyResults  = Series(tertiatyResults).apply(Series)
    return primaryResults,secondaryResults,tertiatyResults

In [ ]:
matchedArtistData      = {item["ArtistID"]: item["Results"] for item in mmg.matchedArtistData}
matchedArtistAlbumData = {item["ArtistID"]: item["Results"] for item in mmg.matchedArtistAlbumData}
primaryResults,secondaryResults,tertiaryResults = getAlbumMatchResults(matchedArtistAlbumData)

tsMatch.update()

primaryCrossMatchResults   = {}
secondaryCrossMatchResults = {}
tertiaryCrossMatchResults  = {}
for db,mmgCross in crossMatchResults.items():
    crossMatchedArtistData      = {item["ArtistID"]: item["Results"] for item in mmgCross.matchedArtistData}
    crossMatchedArtistAlbumData = {item["ArtistID"]: item["Results"] for item in mmgCross.matchedArtistAlbumData}    
    primaryCrossMatchResults[db],secondaryCrossMatchResults[db],tertiaryCrossMatchResults[db] = getAlbumMatchResults(crossMatchedArtistAlbumData)

tsMatch.update()

In [ ]:
stats = DataFrame([primaryResults.count(), secondaryResults.count(), tertiaryResults.count()]).T
stats.columns = ["Primary", "Secondary", "Tertiary"]
stats.T

In [ ]:
if False:
    from pandas import isna,notna

    def exclusiveID(row, db):
        retval = {}
        for i,(key,val) in enumerate(row.iteritems()):
            if isna(val):
                retval[key] = None
                continue
            else:
                retval[key] = val
                for j,(key2,val2) in enumerate(row.iteritems()):
                    if j <= i:
                        continue
                    retval[key2] = None
                break
        return retval


    for db in stats.T.columns:
        if db == dbToUse:
            continue
        df  = DataFrame(primaryResults[db]).join(secondaryResults[db], rsuffix="_2nd").join(tertiaryResults[db], rsuffix="_3rd")
        tmp = df.apply(exclusiveID, db=db, axis=1).apply(Series)
        primaryResults[db],secondaryResults[db],tertiaryResults[db] = tmp["{0}".format(db)],tmp["{0}_2nd".format(db)],tmp["{0}_3rd".format(db)]


    for dbCross in stats.T.columns:
        primaryCrossMatchResults[db],secondaryCrossMatchResults[db],tertiaryCrossMatchResults[db] = getAlbumMatchResults(crossMatchedArtistAlbumData)


In [ ]:
stats = DataFrame([primaryResults.count(), secondaryResults.count(), tertiaryResults.count()]).T
stats.columns = ["Primary", "Secondary", "Tertiary"]
stats.T

## Cross Match Results

In [ ]:
from pandas import notna,isna

matchedResults = {}

noMatchArtists      = {}
newMasterArtists    = {}
multiMasterArtists  = {}
singleMasterArtists = {}

toUse = "primary"
dbToUseResults = {"primary": primaryResults, "secondary": secondaryResults, "tertiary": tertiaryResults}[toUse]
dbCrossMatchResults = {"primary": primaryCrossMatchResults, "secondary": secondaryCrossMatchResults, "tertiary": tertiaryCrossMatchResults}[toUse]


for artistID,artistIDMatches in dbToUseResults.iterrows():
    print("="*25,artistID,"="*77)
    print("   {0: <20}{1: >40} <=> {2: <40}".format("",artistsToMatchData.loc[artistID,"ArtistName"],artistsToMatchData.loc[artistID,"NumAlbums"]))
    
    
    ##############################################################################################################
    # 1) Must match itself
    ##############################################################################################################
    if artistID != artistIDMatches[dbToUse]:
        print("Unsuccessful Self Match For {0}   [{1}]  <=>  [{2}]".format(dbToUse,artistID,artistIDMatches[dbToUse]))
    
    
    ##############################################################################################################
    # 2) Test Cyclical Match
    ##############################################################################################################
    artistIDMatchResults = {}
    print("   {0: <20}{1: >40} <=> {2: <40}{3: <8}{4: <8}".format("DB","DBID","CyclicalID","Match","Known"))
    for matchDB,matchDBID in artistIDMatches.drop([dbToUse]).iteritems():
        if notna(matchDBID):
            print("   {0: <20}{1: >40} <=> ".format(matchDB,matchDBID),end="")
            cyclicalMatchID = dbCrossMatchResults[matchDB].loc[matchDBID,dbToUse]   #### matched DB match to dbToUse
            print("{0: <40}".format(cyclicalMatchID),end="")
            uval = '✓' if cyclicalMatchID == artistID else '✕'
            print("{0: <8}".format(uval),end="")
            
            ##############################################################################################################
            # 2) Test Master DB
            ##############################################################################################################
            knownRow = mmeDF[mmeDF[matchDB] == matchDBID]
            knownIDX = list(knownRow.index)[0] if knownRow.shape[0] > 0 else None
            print(knownIDX)
            
            ##############################################################################################################
            # 3) Save Result
            ##############################################################################################################
            if cyclicalMatchID == artistID:
                if knownIDX is None:
                    if artistIDMatchResults.get("New") is None:
                        artistIDMatchResults["New"] = {}
                    artistIDMatchResults["New"][matchDB] = matchDBID
                else:
                    if artistIDMatchResults.get(knownIDX) is None:
                        artistIDMatchResults[knownIDX] = {}
                    artistIDMatchResults[knownIDX][matchDB] = matchDBID
                    

    print("   {0: <20}{1: >40} <=> ".format("","Result"),end="")
    print("{0: <40}".format(len(artistIDMatchResults)),end="")
    if len(artistIDMatchResults) == 0:
        print("{0: <8}".format('✕'),end="")
        print('✕')
        noMatchArtists[artistID] = artistIDMatchResults
    elif len(artistIDMatchResults) == 1:
        if "New" in artistIDMatchResults.keys():
            print("{0: <8}".format('✓'),end="")
            print('✕')
            newMasterArtists[artistID] = artistIDMatchResults
        else:
            print("{0: <8}".format('✓'),end="")
            print(list(artistIDMatchResults.keys())[0])
            singleMasterArtists[artistID] = artistIDMatchResults
    elif len(artistIDMatchResults) == 2:
        if "New" in artistIDMatchResults.keys():
            print("{0: <8}".format('✓'),end="")
            print([x for x in artistIDMatchResults.keys() if x not in ["New"]][0])
            singleMasterArtists[artistID] = artistIDMatchResults
        else:
            print("{0: <8}".format('✕'),end="")
            print("MultiIDX")            
            multiMasterArtists[artistID] = artistIDMatchResults
    else:
        print("{0: <8}".format('✕'),end="")
        print("MultiIDX")
        multiMasterArtists[artistID] = artistIDMatchResults

In [ ]:
stats = Series({"No Match": noMatchArtists, "New Master": newMasterArtists, "Single Master": singleMasterArtists, "Multi Master": multiMasterArtists})
stats.apply(len).append(Series({"Total": stats.apply(len).sum()}))

# Save To Master Manual Entries Data

## Create Master DataFrames

In [ ]:
####################################################################################################################################
# No DB Matches
####################################################################################################################################
noMasterData = DataFrame(Series(noMatchArtists)).join(artistsToMatchData["ArtistName"])
noMasterData = noMasterData.drop([0], axis=1)
noMasterData = noMasterData.reset_index().rename({"index": dbToUse}, axis=1)


####################################################################################################################################
# New Master Artist Match
####################################################################################################################################
newMasterData = DataFrame({artistID: artistIDMatchResults["New"] for artistID, artistIDMatchResults in newMasterArtists.items()}).T
newMasterData = newMasterData.join(artistsToMatchData["ArtistName"])
newMasterData = newMasterData.reset_index().rename({"index": dbToUse}, axis=1)


####################################################################################################################################
# Single Master Artist Match (+ Possible New DB Matches)
####################################################################################################################################
singleMasterData = {}
for artistID, artistIDMatchResults in singleMasterArtists.items():
    singleMasterKey = None
    if len(artistIDMatchResults) == 2:
        for masterIDX,masterIDXData in artistIDMatchResults.items():
            if masterIDX == "New":
                newD=masterIDXData
            else:
                singleMasterKey=masterIDX
                idxD=masterIDXData
        singleMasterData[singleMasterKey] = {**idxD, **newD}
    else:
        for masterIDX,masterIDXData in artistIDMatchResults.items():
            singleMasterKey = masterIDX
            singleMasterData[singleMasterKey] = masterIDXData
    singleMasterData[singleMasterKey][dbToUse] = artistID
singleMasterData = DataFrame(singleMasterData).T


####################################################################################################################################
# Multi Master Artist Matches
####################################################################################################################################
multiMasterData = DataFrame(Series(multiMasterArtists, name="Multi Master Matches").copy(deep=True))
multiMasterData = multiMasterData.join(artistsToMatchData["ArtistName"])
multiMasterData = multiMasterData.reset_index().rename({"index": dbToUse}, axis=1)


####################################################################################################################################
# Statistics
####################################################################################################################################
tsMatch.update()
stats = Series({"No Match": noMasterData, "New Master": newMasterData, "Single Master": singleMasterData, "Multi Master": multiMasterData})
stats.apply(lambda x: x.shape[0]).append(Series({"Total": stats.apply(len).sum()}))

## Set DataFrames To Master Manual Entries DataFrame

In [ ]:
###################################################################################################################################
# 1) Add New Artists
###################################################################################################################################
print("Adding {0} New Master Artists".format(newMasterData.shape[0]))
print("   Current Number of Master Artists: {0}".format(mmeDF.shape[0]))
newRows = []
for idx,row in newMasterData.iterrows():
    newRow = row[row.notna()]
    newRow.name = str(uuid4())
    newRows.append(newRow)
mmeDF  = mmeDF.append(newRows)
print("   Current Number of Master Artists: {0}".format(mmeDF.shape[0]))


###################################################################################################################################
# 2) Edit Existing Master Artists
###################################################################################################################################
print("Merging {0} Existing Master Artists".format(singleMasterData.shape[0]))
print("   Current Number of Master DB IDs: {0}".format(mmeDF.count().sum()))
errs = {}
for masterIDX,row in singleMasterData.iterrows():
    for db,dbID in row.iteritems():
        if isna(dbID):
            continue
        try:
            if isna(mmeDF.loc[masterIDX,db]):
                mmeDF.loc[masterIDX,db] = dbID
        except:
            errs[masterIDX] = row
            print("Error with {0}/{1}/{2}".format(masterIDX,db,dbID))
print("   Current Number of Master DB IDs: {0}".format(mmeDF.count().sum()))
print("   Found {0} Errors".format(len(errs)))


tsMatch.update()

In [ ]:
errs

# Save Manual Entries Data

In [ ]:
mme.saveData(mmeDF,local=False,fast=True)
if len(multiMasterData) > 0:
    print("Found {0} Multiple Matches".format(multiMasterData.shape[0]))
    io.save(idata=multiMasterData, ifile="multiMasterData{0}.p".format(dbToUse))
tsMatch.stop()

# Near Results Analysis

In [ ]:
from masterManualEntriesUtils import masterManualEntriesUtils
meu = masterManualEntriesUtils()
meu.getDF().tail()

# Short Analysis

In [ ]:
from masterManualEntriesUtils import masterManualEntriesUtils
meu = masterManualEntriesUtils()

In [ ]:
meu.addAlbums()
meu.addCounts()
meu.saveDF()

In [ ]:
dbname    = "RateYourMusic"
discdisc  = mdbGate.getDBDisc(dbname)
discnames = discdisc.getArtistIDToNameData()
discrefs  = discdisc.getArtistIDToRefData()

mmeDF   = meu.getDF()
dbIDs   = mmeDF[dbname][mmeDF[dbname].notna()]
dbLists = dbIDs[dbIDs.apply(lambda x: isinstance(x, list))]
dbRefs  = dbIDs[dbIDs.apply(lambda x: x.startswith("http"))]

In [ ]:
for idx,url in dbRefs.iteritems():
    name    = mmeDF.loc[idx,"ArtistName"]
    
    urlToIDs = {url: discrefs[discrefs.isin([url])] for url in [url]}
    urlToIDs = {url: list(refs.index)[0] if refs.shape[0] == 1 else None for url,refs in urlToIDs.items()}
    urlToIDs = [dbID for dbID in urlToIDs.values() if dbID is not None]
    if len(urlToIDs) == 0:
        continue
        meu.setdbid(idx,dbname,None)
    elif len(urlToIDs) == 1:
        dbID = urlToIDs[0]
        meu.setdbid(idx,dbname,dbID)
    else:
        print("Not sure what to do here: {0}".format(urlToIDs))
        
    continue
    
        
    dbvals = urlToIDs
    
    dnames  = {dbID: discnames.get(dbID) for dbID in dbvals}
    tomatch = Series(dnames)
    tomatch = tomatch[tomatch.notna()]
    retval  = mm.matchNames(tomatch=tomatch, value=name)
    #if retval.max() > 0.7:
    try:
        dbID = retval.idxmax()
    except:
        dbID = None
        continue
    meu.setdbid(idx,dbname,dbID)    
    
    
meu.saveDF()

In [ ]:
mmeDF = meu.getDF()
for col,colData in mmeDF.iteritems():
    lvals = colData[colData.notna()].apply(lambda x: isinstance(x, list))
    print(col,'\t',lvals.sum())

In [ ]:
mmeDF = meu.getDF()
dbname = "RateYourMusic"
dbIDs = mmeDF[dbname][mmeDF[dbname].notna()]
dbLists = dbIDs[dbIDs.apply(lambda x: isinstance(x, list))]
dbRefs = dbIDs[dbIDs.apply(lambda x: x.startswith("http"))]
dbIDLens = dbIDs.apply(len)

In [ ]:
mmeDF = meu.getDF()
dbname = "RateYourMusic"
dbIDs = mmeDF[dbname][mmeDF[dbname].notna()]
dbLists = dbIDs[dbIDs.apply(lambda x: isinstance(x, list))]
dbRefs = dbIDs[dbIDs.apply(lambda x: x.startswith("http"))]
dbIDLens = dbIDs.apply(len)


discdisc = mdbGate.getDBDisc(dbname)
discnames = discdisc.getArtistIDToNameData()
discrefs  = discdisc.getArtistIDToRefData()


util = mdbGate.getDBUtil(dbname)
for idx,lvals in dbLists.iteritems():
    urls = [x for x in lvals if "/artists/" in x]
    if len(urls) == 1:
        meu.setgenid(idx,util.getArtistID(urls[0]))
    elif len(urls) == 0:
        meu.setgenid(idx,None)
    else:
        meu.setgenid(idx,util.getArtistID(urls[0]))        
meu.saveDF()

In [ ]:
idxs = dbIDs[~dbIDs.apply(lambda x: x.isdigit())].index
for idx in idxs:
    print(idx)
    meu.setdbid(idx,dbname,None)
meu.saveDF()
#dbLists

In [ ]:
discrefs

In [ ]:
for idx,urls in dbLists.iteritems():
    for url in urls:
        ids = discrefs[discrefs.isin([url])]
        print(ids,'\t',url)
        #print(discrefs.get(url),'\t',url)

In [ ]:
from matchArtistToDB import masterMatch
mm = masterMatch()
help(mm.matchNames)

In [ ]:
for idx,dbvals in dbLists.iteritems():
    name    = mmeDF.loc[idx,"ArtistName"]
    
    urlToIDs = {url: discrefs[discrefs.isin([url])] for url in dbvals}
    urlToIDs = {url: list(refs.index)[0] if refs.shape[0] == 1 else None for url,refs in urlToIDs.items()}
    urlToIDs = [dbID for dbID in urlToIDs.values() if dbID is not None]
    if len(urlToIDs) == 0:
        meu.setdbid(idx,dbname,None)
        continue
    dbvals = urlToIDs
    
    dnames  = {dbID: discnames.get(dbID) for dbID in dbvals}
    tomatch = Series(dnames)
    tomatch = tomatch[tomatch.notna()]
    retval  = mm.matchNames(tomatch=tomatch, value=name)
    #if retval.max() > 0.7:
    try:
        dbID = retval.idxmax()
    except:
        dbID = None
        continue
    meu.setdbid(idx,dbname,dbID)    

In [ ]:
meu.saveDF()

In [ ]:
dbIDLens.value_counts()

In [ ]:
meu.getMMEByArtist("Ray Miller", "C")
#meu.newArtist("Gus Arnheim's Cocoanut Grove Orchestra", Discogs="708255", MusicBrainz="192274658883207265364491512945592806565")
#meu.setdiscid("ggggggggXXX0024772XXX02", "675269")
#meu.setmbid("ggggggggXXX0024772XXX02", "156444845644491584717939248580577520531")
#meu.nulldeezer("ggggggggXXX0024772XXX02")

In [ ]:
meu.getMMEByMBURL("https://musicbrainz.org/artist/91835282-702e-497d-93d8-506ad97ddf62")
meu.mergeRows("rrrrrrrrXXX0006961XXX1", "rrrrrrrrXXX0006960XXX01")

In [ ]:
for idx,row in mmeDF.loc[dbIDs[dbIDLens >= 9].index].head(8).iterrows():
    print("meu.setdiscid(\'{0}\', \'\')   ## {1}".format(idx, row["ArtistName"]))

In [ ]:
mmeDF.loc[dbIDs[dbIDLens >= 9].index].head(8)

In [ ]:
meu.setdiscid('ttttttttXXX0045626XXX1', '1841998')   ## Tobias Forge
meu.setrymid("ttttttttXXX0045626XXX1", "[Artist1322647]")
meu.saveDF()

In [ ]:
mmeDF.loc[dbIDs[dbIDLens == 1].index]

In [ ]:
mmeDF = meu.getDF()

In [ ]:
colLens = {col: colData[colData.notna()].apply(len).value_counts() for col,colData in mmeDF[mdbGate.dbs].iteritems()}

In [ ]:
colLens

In [ ]:
#meu = masterManualEntriesUtils()
mmeDF = mme.getData()
#mmeDF

In [ ]:
db = "RateYourMusic"
dbData = mmeDF[db][(mmeDF[db].notna())]

In [ ]:
from dbArtistsID import artistIDDiscogs, artistIDAllMusic, artistIDMusicBrainz, artistIDGenius, artistIDDeezer, artistIDLastFM
utils = {"Discogs": artistIDDiscogs(), "AllMusic": artistIDAllMusic(), "MusicBrainz": artistIDMusicBrainz(), "Genius": artistIDGenius(), "Deezer": artistIDDeezer(), "LastFM": artistIDLastFM()}

In [ ]:
disc = mdbGate.getDBDisc(db)
refData = disc.getArtistIDToRefData()

In [ ]:
refData.name = "RateYourMusic"
refData = DataFrame(refData).reset_index()

In [ ]:
mmeDF = mme.getData()

In [ ]:
mdbData = masterDBData(dtype="Search")
mdbData.loadArtists()
dbCols = mdbData.dbs

def getNumAlbums(dbID,db):
    return mdbData.getArtistDBNumAlbumsFromID(db,dbID)

ts = timestat("Getting Albums For Each DBID")
mmeDFAlbums = {db: mmeDF[db].apply(getNumAlbums, db=db) for db in dbCols}
ts.stop()
#mmeDFRank   = {db: dbIDs[dbIDs.notna()].rank(pct=True) for db,dbIDs in mmeDFAlbums.items()}
#rank = DataFrame(mmeDFRank).fillna(0.0).mean(axis=1).rank(pct=True)

In [ ]:
dbIDs[dbIDs.notna()].apply(lambda x: len(str(x))).value_counts()

In [ ]:
for db,dbIDs in mmeDFAlbums.items():
    print(db)
    _ = dbIDs[dbIDs.notna()].rank(pct=True)


In [ ]:
meu = masterManualEntriesUtils()
meu.addAlbums()
meu.addCounts()

In [ ]:
from pandas import merge
mval = merge(DataFrame(dbData[dbData.apply(len) >= 8]).reset_index(), refData, on="RateYourMusic", how="left")
meu = masterManualEntriesUtils()
for idx,row in mval[mval["index_y"].notna()].iterrows():
    meu.setrymid(row["index_x"], row["index_y"])
meu.saveDF()

In [ ]:
meu = masterManualEntriesUtils()
for idx,lfmVal in lfmData[lfmData.apply(len) >= 15].iteritems():
    meu.nulllastfm(idx)
meu.saveDF()

In [ ]:
#lfmToDo = lfmData[lfmData.apply(lambda x: x.startswith("http") if isinstance(x, str) else False)]
lfmToDo = lfmData[lfmData.apply(lambda x: isinstance(x, list))]
lfmToDo

In [ ]:
util = mdbGate.getDBUtil("LastFM")
lfmToDo.apply(util.getArtistID)

In [ ]:
meu = masterManualEntriesUtils()
for idx,lfmVal in lfmToDo.iteritems():
    meu.nulllastfm(idx)
    continue
    if Series(lfmVal).nunique() == 1:
        meu.setdbid(idx,"LastFM",util.getArtistID(lfmVal[0]))
meu.saveDF()

In [ ]:
util = mdbGate.getDBUtil("LastFM")
meu = masterManualEntriesUtils()
for idx,genID in lfmToDo.apply(util.getArtistID).iteritems():
    meu.setdbid(idx,"LastFM",genID)
meu.saveDF()

In [ ]:
for col in mmeDF.columns:
    if col in ["ArtistName"]:
        continue
    mmeDF.loc[mmeDF[col].isna(), col] = None
mme.saveData(mmeDF)

In [ ]:
meu.saveDF()

In [ ]:
import Levenshtein
from uuid import uuid4
from pandas import Series
from functools import partial
from masterDBGate import masterDBGate

from dbArtistsID import artistIDMusicBrainz
aid = artistIDMusicBrainz()

def getRawData(db, dbID):
    dbmap = masterDBGate().getDBMap(db)
    retval = dbmap['Artist'].getData(dbmap['Disc'].getRawFilename(dbID))
    return retval

def getmbid(url):
    return aidmb.getArtistID(url)

def getMMEByMBURL(url):
    return getMMEByID('MusicBrainz', getmbid(url))

def getMMEByArtist(name, match="E"):
    if match == "E":
        return mmeDF[mmeDF["ArtistName"] == name]
    elif match == "C":
        return mmeDF[mmeDF["ArtistName"].str.contains(name)]
    else:
        idxs = mmeDF["ArtistName"].apply(lambda x: Levenshtein.ratio(x.upper(), name.upper())).sort_values(ascending=False).head(int(match)).index
        return mmeDF.loc[idxs]
    
def getMMEByID(db, dbID):
    return mmeDF[mmeDF[db] == str(dbID)]

def setdbid(idx, db, dbID):
    mmeDF.loc[idx, db] = str(dbID)
    
def setrymid(idx, dbID):
    if dbID.startswith('[Artist') and dbID.endswith(']'):
        dbID = dbID[7:-1]
    elif dbID.startswith('Artist'):
        dbID = dbID[6:]    
    setdbid(idx, "RateYourMusic", dbID)
    
def setgenid(idx, dbID):
    setdbid(idx, "Genius", dbID)
    
def setdiscid(idx, dbID):
    setdbid(idx, "Discogs", dbID)
    
def setamid(idx, dbID):
    setdbid(idx, "AllMusic", dbID)
    
def setaotyid(idx, dbID):
    setdbid(idx, "AlbumOfTheYear", dbID)
    
def setname(idx, name):
    setdbid(idx, "ArtistName", name)

def newRow(row):
    row = Series(row)
    row.name = str(uuid4())
    return row

def addRow(name, **kwargs):
    row = {"ArtistName": name}
    row.update({k: v for k,v in kwargs.items() if k in mmeDF.columns})
    if len(row) > 1:
        return newRow(row)
        #mmeDF = mmeDF.append(newRow(row), inplace=True)

def dropRow(idx):
    mmeDF.drop([idx], axis=0, inplace=True)
    
def newArtist(name, **kwargs):
    row = {"ArtistName": name}
    row.update({k: v for k,v in kwargs.items() if k in mmeDF.columns})
    if len(row) > 1:
        return newRow(row)
    else:
        print("Need valid db")

def mergeRows(idx1, idx2):
    for db,dbID in mmeDF.loc[idx2][mmeDF.loc[idx2].notna()].iteritems():
        if isna(mmeDF.loc[idx1,db]):
            setdbid(idx1, db, dbID)
    mmeDF.drop([idx2], axis=0, inplace=True)

In [ ]:
artistsToMatchData

In [ ]:
i = 47
artistName = artistsToMatchData.iloc[i]["ArtistName"]
print(artistsToMatchData.iloc[i])
db = "AlbumOfTheYear"
#print(artistName)
tmp = getMMEByArtist(artistName,1)

print("mmeDF=mmeDF.append(newArtist(\"{0}\", RateYourMusic='', AlbumOfTheYear='{1}'))".format(artistName, artistsToMatchData.iloc[i].name))

tmp

In [ ]:
mmeDF=mmeDF.append(newArtist("Pro Zay", RateYourMusic='1333844', AlbumOfTheYear='64170'))

In [ ]:
mme.saveData(mmeDF, local=False, fast=True)

In [ ]:
mmeDF.drop(['d910bae3-2db1-48d5-8ebd-bc0390dfec08'], axis=0, inplace=True)